# Eye-Tracking Dyslexia Detection — Pipeline Showdown
## Baseline (Strawman) vs. REMODNAV (Challenger)

This notebook implements a **two-pipeline comparison** on the ETDD-70 dataset:

| Pipeline | Description |
|----------|-------------|
| **Strawman** | I-DT fixation + I-VT saccade detection. No blink interpolation — missing data stays missing. Left & right eyes processed separately. Standard naive approach. |
| **REMODNAV** | Off-the-shelf peer-reviewed event detector (Dar et al. 2021). Adaptive thresholds, handles noise automatically. Fed raw Tobii arrays directly. |

Both pipelines extract the **same 41 tabular features** and run through the **same classifiers (RF, SVM) under LOOCV**.


## 1. Imports

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import signal as scipy_signal
from scipy.stats import mannwhitneyu

from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut, cross_validate
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
print("Imports OK")


Imports OK


## 2. Configuration

In [2]:
# ============================================================
# CONFIGURATION — adjust paths before running
# ============================================================
from google.colab import drive
import numpy as np

drive.mount('/content/drive')

DATA_FOLDER = '/content/drive/MyDrive/TA_Project/dataset/data'
METADATA_FILE = '/content/drive/MyDrive/TA_Project/dataset/dyslexia_class_label.csv'
TARGET_TASK = "T5_Pseudo_Text"

# ------------------------------------------------------------
# 1. HARDWARE SPECS (CRITICAL LIABILITY)
# ------------------------------------------------------------
# WARNING: If ETDD-70 is 60Hz or 250Hz, 1000Hz breaks all velocity math.
# You MUST verify this with the dataset documentation or raw timestamps.
SAMPLING_RATE       = 250   # Hz (Placeholder — Verify this)

# ------------------------------------------------------------
# 2. I-DT / I-VT THRESHOLDS (DIAGNOSTIC & PAPER-ALIGNED)
# ------------------------------------------------------------
# The ETDD-70 authors specifically had to drop this to 40ms to catch
# children's fixations. Do not change this back to 100ms.
FIXATION_MIN_MS     = 40     # ms
# Widened to 3.0 so I-DT survives spatial hardware noise.
# Once you verify screen geometry, you can tighten this back to ~1.0.
FIXATION_MAX_DISP   = 1.0    # visual degrees
SACCADE_MIN_VEL     = 15     # deg/s

# Regression filter (leftward saccades within a reading line)
REGRESSION_THRESHOLD   = 0    # pixel — leftward dx
REGRESSION_MAX_AMP_DEG = 3.0  # deg — filters out large return sweeps

# ------------------------------------------------------------
# 3. SCREEN GEOMETRY (CRITICAL LIABILITY)
# ------------------------------------------------------------
# If they used an iPad or a 24-inch monitor, this math is completely wrong.
SCREEN_WIDTH_CM  = 47.0
SCREEN_WIDTH_PX  = 1680      # Corrected from 1920
VIEWING_DIST_CM  = 60    # (Placeholder — Verify this)

PX_PER_DEG = (SCREEN_WIDTH_PX / SCREEN_WIDTH_CM) * (VIEWING_DIST_CM * np.tan(np.deg2rad(1)))

# Sliding-window segmentation
WINDOW_SIZE_SEC = 4
WINDOW_OVERLAP  = 0.5

# Feature selection & CV
N_TOP_FEATURES  = 20     # top-K ANOVA features per Pipeline
USE_LOOCV       = True   # LOOCV for unbiased n=70 estimate

print(f"PX_PER_DEG : {PX_PER_DEG:.2f}")
print(f"CV strategy: {'LOOCV' if USE_LOOCV else 'K-Fold'}")

Mounted at /content/drive
PX_PER_DEG : 37.44
CV strategy: LOOCV


## 3. Data Loading

In [20]:
def load_raw_subject(filepath):
    """
    Load one subject CSV. Returns DataFrame with columns:
      avg_x, avg_y, gaze_x_left, gaze_x_right, gaze_y_left, gaze_y_right, timestamp
    NaN rows = blinks — kept as-is (Strawman pipeline needs them).
    """
    with open(filepath, 'r') as f:
        first_line = f.readline()
    sep = ';' if ';' in first_line else ','
    df = pd.read_csv(filepath, sep=sep, low_memory=False)

    # Fix European decimal notation
    for col in df.columns:
        if df[col].dtype == object:
            try:
                df[col] = df[col].str.replace(',', '.', regex=False).astype(float)
            except (ValueError, AttributeError):
                pass

    def find_col(*kws):
        for col in df.columns:
            if all(k.lower() in col.lower() for k in kws):
                return col
        return None

    lx = find_col('gaze_x_left')  or find_col('left', 'x')
    rx = find_col('gaze_x_right') or find_col('right', 'x')
    ly = find_col('gaze_y_left')  or find_col('left', 'y')
    ry = find_col('gaze_y_right') or find_col('right', 'y')
    ts = find_col('timestamp')    or find_col('time')

    if not (lx and rx and ly and ry):
        raise ValueError(f"Gaze columns not found in: {filepath}")

    result = pd.DataFrame({
        'gaze_x_left' : df[lx].astype(float),
        'gaze_x_right': df[rx].astype(float),
        'gaze_y_left' : df[ly].astype(float),
        'gaze_y_right': df[ry].astype(float),
    })
    result['avg_x'] = (result['gaze_x_left']  + result['gaze_x_right']) / 2
    result['avg_y'] = (result['gaze_y_left']  + result['gaze_y_right']) / 2
    result['timestamp'] = df[ts].astype(float) if ts else np.arange(len(result)) * (1000.0 / SAMPLING_RATE)
    return result


def load_all_subjects():
    print("Loading metadata...")
    meta = pd.read_csv(METADATA_FILE, sep=',')
    meta['subject_id'] = meta['subject_id'].astype(str)

    subject_dfs, labels = {}, {}
    files = os.listdir(DATA_FOLDER)
    print(f"Scanning {len(files)} files for task: {TARGET_TASK}")

    for filename in files:
        if TARGET_TASK in filename and 'raw.csv' in filename:
            subj_id = filename.split('_')[1]
            row = meta[meta['subject_id'] == subj_id]
            if row.empty:
                continue
            label = row['class_id'].values[0]
            try:
                df = load_raw_subject(os.path.join(DATA_FOLDER, filename))
                if len(df.dropna()) > 100:
                    subject_dfs[subj_id] = df
                    labels[subj_id] = label
            except Exception as e:
                print(f"  ⚠ Skip {filename}: {e}")

    print(f"✓ Loaded {len(subject_dfs)} valid subjects")
    print(f"  Class distribution:\n{pd.Series(labels).value_counts().to_string()}")
    return subject_dfs, labels


subject_dfs, labels = load_all_subjects()


Loading metadata...
Scanning 837 files for task: T5_Pseudo_Text
✓ Loaded 70 valid subjects
  Class distribution:
1    35
0    35


## 4. Pipeline A — Strawman (I-DT + I-VT Baseline)

**Rules:**
- I-DT for fixations, I-VT for saccades (Holmqvist et al. 2011)
- **No blink interpolation** — NaN rows are skipped, not filled
- Left and right eyes processed **separately**, features averaged at the end
- Standard naive approach that serves as the scientific floor


In [4]:
# ── Strawman helpers ───────────────────────────────────────────────────────

from scipy.signal import savgol_filter

def compute_velocity_single(x, y, sampling_rate=SAMPLING_RATE):
    """Velocity (deg/s) for a single eye trace. NaN-safe with Savitzky-Golay smoothing."""
    dt = 1.0 / sampling_rate

    # 1. Calculate raw velocity
    dx = np.diff(x, prepend=x[0]) / PX_PER_DEG
    dy = np.diff(y, prepend=y[0]) / PX_PER_DEG
    vel = np.sqrt(dx**2 + dy**2) / dt

    # 2. Smooth the velocity (Savitzky-Golay filter)
    # Window length of 7, polynomial order of 2 is standard for eye tracking.
    # We must only smooth the valid (non-NaN) segments.
    valid = ~np.isnan(vel)
    if valid.sum() > 7:
        # Interpolate NaNs temporarily just for the filter
        vel_interp = pd.Series(vel).interpolate(method='linear').bfill().ffill().values
        vel_smooth = savgol_filter(vel_interp, window_length=7, polyorder=2)

        # Put the NaNs back where they belong
        vel_smooth[~valid] = np.nan
        return vel_smooth
    else:
        return vel


def detect_fixations_idt_single(x, y, ts,
                                  min_duration_ms=FIXATION_MIN_MS,
                                  max_dispersion_deg=FIXATION_MAX_DISP):
    """
    I-DT on a single (x, y) trace. NaN samples are skipped (blinks stay missing).
    Returns list of fixation dicts.
    """
    disp_px   = max_dispersion_deg * PX_PER_DEG
    min_samps = int((min_duration_ms / 1000.0) * SAMPLING_RATE)
    fixations = []
    n = len(x)
    i = 0

    while i < n:
        if np.isnan(x[i]) or np.isnan(y[i]):
            i += 1
            continue

        win_start = i
        j = min(i + min_samps, n)
        wx = x[win_start:j];  wy = y[win_start:j]
        valid = ~(np.isnan(wx) | np.isnan(wy))
        if valid.sum() < max(2, min_samps // 2):
            i += 1
            continue

        disp = (np.nanmax(wx) - np.nanmin(wx)) + (np.nanmax(wy) - np.nanmin(wy))
        if disp > disp_px:
            i += 1
            continue

        # Extend window
        while j < n:
            if np.isnan(x[j]) or np.isnan(y[j]):
                break
            nd = ((np.nanmax(x[win_start:j+1]) - np.nanmin(x[win_start:j+1])) +
                  (np.nanmax(y[win_start:j+1]) - np.nanmin(y[win_start:j+1])))
            if nd > disp_px:
                break
            j += 1

        dur = ts[j-1] - ts[win_start] if j > win_start else 0
        if dur >= min_duration_ms:
            sx = x[win_start:j];  sy = y[win_start:j]
            fixations.append({
                'duration_ms'  : dur,
                'centroid_x'   : np.nanmean(sx),
                'centroid_y'   : np.nanmean(sy),
                'dispersion_px': (np.nanmax(sx)-np.nanmin(sx) + np.nanmax(sy)-np.nanmin(sy)),
            })
        i = j

    return fixations


def detect_saccades_ivt_single(x, y, ts):
    """
    I-VT on a single (x, y) trace. NaN samples excluded from saccade boundaries.
    Returns list of saccade dicts including is_regression flag.
    """
    vel  = compute_velocity_single(x, y)
    is_s = (vel > SACCADE_MIN_VEL) & ~np.isnan(vel)
    saccades, in_s, start = [], False, 0

    for i in range(len(is_s)):
        if is_s[i] and not in_s:
            in_s = True;  start = i
        elif not is_s[i] and in_s:
            in_s = False;  end = i - 1
            if end >= start and not (np.isnan(x[end]) or np.isnan(x[start])):
                dx  = x[end] - x[start]
                dy  = y[end] - y[start]
                amp = np.sqrt((dx/PX_PER_DEG)**2 + (dy/PX_PER_DEG)**2)
                saccades.append({
                    'duration_ms'   : ts[end] - ts[start],
                    'amplitude_deg' : amp,
                    'peak_velocity' : np.nanmax(vel[start:end+1]),
                    'dx_px'         : dx,
                    'is_regression' : (dx < REGRESSION_THRESHOLD and amp < REGRESSION_MAX_AMP_DEG),
                })
    return saccades


def _event_features(fixations, saccades, total_ms, prefix=''):
    """Compute the 41-feature vector from event lists."""
    feats = {}
    p = prefix

    # Fixation
    n_fix = len(fixations)
    if n_fix > 0:
        fd = [f['duration_ms']   for f in fixations]
        fp = [f['dispersion_px'] for f in fixations]
        fx = [f['centroid_x']    for f in fixations]
        fy = [f['centroid_y']    for f in fixations]
        feats.update({
            f'{p}fix_count'         : n_fix,
            f'{p}fix_rate_pm'       : n_fix / (total_ms/60000) if total_ms > 0 else 0,
            f'{p}fix_dur_mean'      : np.mean(fd),
            f'{p}fix_dur_std'       : np.std(fd),
            f'{p}fix_dur_median'    : np.median(fd),
            f'{p}fix_dur_min'       : np.min(fd),
            f'{p}fix_dur_max'       : np.max(fd),
            f'{p}fix_dur_q25'       : np.percentile(fd, 25),
            f'{p}fix_dur_q75'       : np.percentile(fd, 75),
            f'{p}fix_disp_mean'     : np.mean(fp),
            f'{p}fix_disp_std'      : np.std(fp),
            f'{p}fix_cx_std'        : np.std(fx),
            f'{p}fix_cy_std'        : np.std(fy),
            f'{p}fix_time_prop'     : sum(fd) / total_ms if total_ms > 0 else 0,
        })
    else:
        for k in ['fix_count','fix_rate_pm','fix_dur_mean','fix_dur_std','fix_dur_median',
                  'fix_dur_min','fix_dur_max','fix_dur_q25','fix_dur_q75',
                  'fix_disp_mean','fix_disp_std','fix_cx_std','fix_cy_std','fix_time_prop']:
            feats[f'{p}{k}'] = 0.0

    # Saccade
    n_sacc = len(saccades)
    if n_sacc > 0:
        sd = [s['duration_ms']   for s in saccades]
        sa = [s['amplitude_deg'] for s in saccades]
        sv = [s['peak_velocity'] for s in saccades]
        feats.update({
            f'{p}sacc_count'         : n_sacc,
            f'{p}sacc_rate_pm'       : n_sacc / (total_ms/60000) if total_ms > 0 else 0,
            f'{p}sacc_dur_mean'      : np.mean(sd),
            f'{p}sacc_dur_std'       : np.std(sd),
            f'{p}sacc_amp_mean'      : np.mean(sa),
            f'{p}sacc_amp_std'       : np.std(sa),
            f'{p}sacc_amp_median'    : np.median(sa),
            f'{p}sacc_pv_mean'       : np.mean(sv),
            f'{p}sacc_pv_std'        : np.std(sv),
            f'{p}sacc_fix_ratio'     : n_sacc / n_fix if n_fix > 0 else 0,
            f'{p}sacc_len_mean_px'   : np.mean([abs(s['dx_px']) for s in saccades]),
        })
    else:
        for k in ['sacc_count','sacc_rate_pm','sacc_dur_mean','sacc_dur_std',
                  'sacc_amp_mean','sacc_amp_std','sacc_amp_median',
                  'sacc_pv_mean','sacc_pv_std','sacc_fix_ratio','sacc_len_mean_px']:
            feats[f'{p}{k}'] = 0.0

    # Regression
    if n_sacc > 0:
        regr = [s for s in saccades if s['is_regression']]
        prog = [s for s in saccades if not s['is_regression']]
        nr   = len(regr)
        feats[f'{p}regr_count']      = nr
        feats[f'{p}regr_proportion'] = nr / n_sacc
        feats[f'{p}regr_amp_mean']   = np.mean([abs(r['amplitude_deg']) for r in regr]) if nr > 0 else 0.0
        feats[f'{p}regr_amp_std']    = np.std([abs(r['amplitude_deg'])  for r in regr]) if nr > 0 else 0.0
        feats[f'{p}regr_dur_mean']   = np.mean([r['duration_ms']        for r in regr]) if nr > 0 else 0.0
        prog_amp = np.mean([p_['amplitude_deg'] for p_ in prog]) if prog else 0.0
        feats[f'{p}prog_amp_mean']   = prog_amp
        feats[f'{p}regr_prog_ratio'] = feats[f'{p}regr_amp_mean'] / prog_amp if prog_amp > 0 else 0.0
    else:
        for k in ['regr_count','regr_proportion','regr_amp_mean','regr_amp_std',
                  'regr_dur_mean','prog_amp_mean','regr_prog_ratio']:
            feats[f'{p}{k}'] = 0.0

    return feats

def extract_features_strawman(df, subject_id=None):
    """
    Strawman feature extraction:
    - No blink interpolation; NaN = missing
    - Process left eye separately, right eye separately
    - Average the per-eye feature vectors
    """
    # FIX 1: Ignore Tobii hardware ticks. Force a perfect ms timeline based on the sampling rate.
    # A 4000-sample window at 1000Hz is exactly 4000ms.
    ts = np.arange(len(df)) * (1000.0 / SAMPLING_RATE)
    total_ms = ts[-1] - ts[0] if len(ts) > 1 else 0

    all_feats = {}
    all_feats['subject_id'] = subject_id

    # Global blink metric (from averaged trace)
    n_blinks = df['avg_x'].isna().sum()
    all_feats['blink_count']   = n_blinks
    all_feats['blink_rate_pm'] = n_blinks / (total_ms / 60000) if total_ms > 0 else 0

    # Per-eye processing
    eye_feats = []
    for eye, xcol, ycol in [('L', 'gaze_x_left', 'gaze_y_left'),
                            ('R', 'gaze_x_right', 'gaze_y_right')]:

        # FIX 2: Apply a 5-sample median filter to neutralize high-frequency Tobii spikes.
        # Without this, classical I-DT max/min dispersion breaks instantly on 1000Hz data.
        ex = pd.Series(df[xcol].values).rolling(window=5, min_periods=1, center=True).median().values
        ey = pd.Series(df[ycol].values).rolling(window=5, min_periods=1, center=True).median().values

        # NaN-safe: keep blinks as NaN, no interpolation
        valid = ~(np.isnan(ex) | np.isnan(ey))
        if valid.sum() < 50:
            continue   # too few valid samples for this eye

        # Velocity / gaze stats on valid samples only
        vel = compute_velocity_single(ex, ey)
        vel_clean = vel[valid & (vel < 3000)]

        fix = detect_fixations_idt_single(ex, ey, ts)
        sac = detect_saccades_ivt_single(ex, ey, ts)

        ef = _event_features(fix, sac, total_ms, prefix='')

        # Velocity stats
        ef['vel_mean']   = np.mean(vel_clean)   if len(vel_clean) > 0 else 0.0
        ef['vel_std']    = np.std(vel_clean)    if len(vel_clean) > 0 else 0.0
        ef['vel_median'] = np.median(vel_clean) if len(vel_clean) > 0 else 0.0
        ef['vel_q95']    = np.percentile(vel_clean, 95) if len(vel_clean) > 0 else 0.0

        # Gaze range
        ef['gaze_x_range'] = np.nanmax(ex) - np.nanmin(ex)
        ef['gaze_y_range'] = np.nanmax(ey) - np.nanmin(ey)
        ef['total_reading_time_ms'] = total_ms

        eye_feats.append(ef)

    if not eye_feats:
        return None

    # Average left+right eye features
    feat_keys = list(eye_feats[0].keys())
    for k in feat_keys:
        all_feats[k] = np.mean([ef[k] for ef in eye_feats])

    return all_feats


print("Strawman functions defined.")


Strawman functions defined.


## 5. Run Strawman Feature Extraction

In [5]:
from tqdm.notebook import tqdm

window_samples = int(WINDOW_SIZE_SEC * SAMPLING_RATE)
step_size      = int(window_samples * (1 - WINDOW_OVERLAP))

all_strawman_feats = []
failed_strawman    = []

for subj_id, df in tqdm(subject_dfs.items(), desc="Processing Subjects"):
    try:
        n_samples = len(df)
        for start_idx in range(0, n_samples - window_samples + 1, step_size):
            end_idx   = start_idx + window_samples
            window_df = df.iloc[start_idx:end_idx].copy().reset_index(drop=True)

            # Skip if > 30% blink in either eye
            if (window_df['gaze_x_left'].isna().sum()  > window_samples * 0.3 or
                window_df['gaze_x_right'].isna().sum() > window_samples * 0.3):
                continue

            feat = extract_features_strawman(window_df, subject_id=subj_id)
            if feat is None:
                continue
            feat['label']        = labels[subj_id]
            feat['window_start'] = start_idx
            all_strawman_feats.append(feat)

    except Exception as e:
        print(f"  ⚠ Error subject {subj_id}: {e}")
        failed_strawman.append(subj_id)

df_strawman = pd.DataFrame(all_strawman_feats)
print(f"Strawman windows  : {len(df_strawman)}")
print(f"Feature columns   : {df_strawman.shape[1] - 3}")   # -subject_id, -label, -window_start
print(f"Class distribution: {df_strawman['label'].value_counts().to_dict()}")
if failed_strawman:
    print(f"Failed subjects   : {failed_strawman}")


Processing Subjects:   0%|          | 0/70 [00:00<?, ?it/s]

Strawman windows  : 7037
Feature columns   : 41
Class distribution: {1: 4651, 0: 2386}


In [6]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut, cross_validate
import numpy as np

def run_knn_only(df_feats):
    # 1. Isolate features, labels, and groups
    drop_cols = {'label', 'subject_id', 'window_start'}
    feat_cols = [c for c in df_feats.columns if c not in drop_cols]

    X = df_feats[feat_cols].values
    y = df_feats['label'].values
    groups = df_feats['subject_id'].values

    # 2. Clean NaN values
    col_mask = ~np.isnan(X).all(axis=0)
    X = X[:, col_mask]
    X = np.nan_to_num(X, nan=0.0)

    logo = LeaveOneGroupOut()

    # Test different neighborhood sizes to find the sweet spot
    neighbors_to_test = [3, 5, 7, 9, 11, 15, 18, 21, 25, 28, 30]

    print(f"\n{'='*65}")
    print(f"  k-Nearest Neighbors (kNN) Evaluation  |  X: {X.shape}")
    print(f"{'='*65}")

    for k in neighbors_to_test:
        # CRITICAL: StandardScaler ensures large numbers don't overpower small decimals
        # weights='distance' gives closer neighbors more voting power, which helps with noisy ET data
        pipe = make_pipeline(
            StandardScaler(),
            KNeighborsClassifier(n_neighbors=k, weights='distance')
        )

        # We use standard accuracy and macro F1
        scores = cross_validate(
            pipe, X, y, groups=groups, cv=logo,
            scoring={'accuracy': 'accuracy', 'f1': 'f1_macro'},
            return_train_score=True, n_jobs=-1
        )

        acc  = scores['test_accuracy'].mean()
        f1   = scores['test_f1'].mean()
        gap  = scores['train_accuracy'].mean() - acc

        print(f"  kNN (k={k:<2})          Acc={acc:.4f}  F1={f1:.4f}  gap={gap:+.4f}")

In [7]:
from sklearn.model_selection import LeaveOneOut, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd

# 1. CREATE THE MACRO DATASET (1 Row per Subject)
# We drop 'window_start' because it has no meaning at the macro level
print("Aggregating 7,037 windows into 70 subject-level summaries...")
df_macro = df_strawman.drop(columns=['window_start']).groupby('subject_id').mean().reset_index()
df_macro['label'] = df_macro['label'].astype(int)

# 2. ISOLATE FEATURES
feat_cols = [c for c in df_macro.columns if c not in {'label', 'subject_id'}]
X_macro = df_macro[feat_cols].values
y_macro = df_macro['label'].values

# Clean NaNs
col_mask = ~np.isnan(X_macro).all(axis=0)
X_macro = X_macro[:, col_mask]
X_macro = np.nan_to_num(X_macro, nan=0.0)

print(f"\n{'='*65}")
print(f"  MACRO BENCHMARK (Apples-to-Apples)  |  X: {X_macro.shape}")
print(f"{'='*65}")

# 3. BUILD MACRO CLASSIFIERS
# Standard scaling is absolutely mandatory here
macro_classifiers = {
    "Logistic Regression (L2)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced')),
    "SVM (RBF)":                make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True, class_weight='balanced')),
    "kNN (k=5)":                make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5, weights='distance')),
    "kNN (k=9)":                make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9, weights='distance'))
}

# 4. LEAVE-ONE-OUT CV (Since n=70, LOOCV naturally leaves one subject out)
loo = LeaveOneOut()

for name, pipe in tqdm(macro_classifiers.items(), desc="Evaluating Macro Models"):
    scores = cross_validate(
        pipe, X_macro, y_macro, cv=loo,
        scoring={'accuracy': 'accuracy'},
        return_train_score=True, n_jobs=-1
    )

    acc  = scores['test_accuracy'].mean()
    gap  = scores['train_accuracy'].mean() - acc

    tqdm.write(f"  {name:<26} Acc={acc:.4f}   gap={gap:+.4f}")

Aggregating 7,037 windows into 70 subject-level summaries...

  MACRO BENCHMARK (Apples-to-Apples)  |  X: (70, 41)


Evaluating Macro Models:   0%|          | 0/4 [00:00<?, ?it/s]

  Logistic Regression (L2)   Acc=0.8143   gap=+0.0652
  SVM (RBF)                  Acc=0.7286   gap=+0.1658
  kNN (k=5)                  Acc=0.5714   gap=+0.4286
  kNN (k=9)                  Acc=0.6000   gap=+0.4000


In [8]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
import pandas as pd
import numpy as np

# 1. Scale the data (Mandatory for Logistic Regression to compare features fairly)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_macro)

# 2. Define the core algorithm (same as your 81.4% winner)
lr = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)

# 3. Initialize the ruthless pruning algorithm
print("Running Recursive Feature Elimination (RFECV)...")
rfecv = RFECV(
    estimator=lr,
    step=1,              # Drop exactly 1 weakest feature per round
    cv=LeaveOneOut(),    # Strictly use the LOOCV that generated the 81.4%
    scoring='accuracy',
    n_jobs=-1
)

# 4. Fit the pruner
rfecv.fit(X_scaled, y_macro)

peak_acc = rfecv.cv_results_['mean_test_score'].max()
print(f"\nOptimal number of features: {rfecv.n_features_} (out of {X_macro.shape[1]})")
print(f"Peak Accuracy achieved    : {peak_acc:.4f}")

# 5. Extract the winning features and their actual mathematical weights
selected_mask = rfecv.support_
optimal_features = np.array(feat_cols)[selected_mask]

# Re-fit the model one final time on just the optimal features to grab the final weights
lr.fit(X_scaled[:, selected_mask], y_macro)
weights = lr.coef_[0]

# 6. Display the final signature
df_weights = pd.DataFrame({
    'Feature': optimal_features,
    'Weight': weights,
    'Absolute_Weight': np.abs(weights)
}).sort_values(by='Absolute_Weight', ascending=False)

print("\n=== THE MATHEMATICAL SIGNATURE OF DYSLEXIA ===")
print("  (Positive weight = predicts Dyslexia)")
print("  (Negative weight = predicts Normal Reader)\n")
print(df_weights[['Feature', 'Weight']].to_string(index=False))

Running Recursive Feature Elimination (RFECV)...

Optimal number of features: 9 (out of 41)
Peak Accuracy achieved    : 0.8571

=== THE MATHEMATICAL SIGNATURE OF DYSLEXIA ===
  (Positive weight = predicts Dyslexia)
  (Negative weight = predicts Normal Reader)

      Feature    Weight
   fix_cx_std -1.278345
 gaze_x_range -1.205669
   regr_count  0.987037
 sacc_pv_mean  0.982661
 fix_disp_std  0.928312
prog_amp_mean  0.837789
      vel_q95 -0.814409
  fix_dur_q25  0.807874
   vel_median  0.618573


# 9. creating images for deep learning model

adding improvement for current training model

In [9]:
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import numpy as np
import os
from tqdm.notebook import tqdm

IMAGE_DIR = "/content/drive/MyDrive/TA_Project/dataset/improved_images/cnn_images"
os.makedirs(IMAGE_DIR, exist_ok=True)

SCREEN_W = 1680
SCREEN_H = 1050

def generate_exaggerated_signature(df_raw, subject_id, label):
    x = df_raw[['gaze_x_left', 'gaze_x_right']].mean(axis=1).rolling(window=5, center=True).median().values
    y = df_raw[['gaze_y_left', 'gaze_y_right']].mean(axis=1).rolling(window=5, center=True).median().values

    valid = ~(np.isnan(x) | np.isnan(y))
    x, y = x[valid], y[valid]

    if len(x) < 50:
        return False

    fig, ax = plt.subplots(figsize=(SCREEN_W/100, SCREEN_H/100), dpi=100)
    fig.patch.set_facecolor('black')
    ax.set_facecolor('black')
    ax.set_xlim(0, SCREEN_W)
    ax.set_ylim(SCREEN_H, 0)
    ax.axis('off')

    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    dx = np.diff(x)

    # Protokol Red/White: Garis progresif normal (dx >= -5) dibuat transparan penuh (alpha=0.0)
    # Regresi dibuat merah solid.
    colors = [(1.0, 0.0, 0.0, 0.9) if delta < -5 else (0.0, 0.0, 0.0, 0.0) for delta in dx]

    lc = LineCollection(segments, colors=colors, linewidths=3)
    ax.add_collection(lc)

    # Titik fiksasi putih
    ax.scatter(x, y, color='white', s=8, alpha=0.15, zorder=3)

    label_folder = "dyslexic" if label == 1 else "normal"
    save_path = os.path.join(IMAGE_DIR, label_folder)
    os.makedirs(save_path, exist_ok=True)

    plt.savefig(f"{save_path}/subj_{subject_id}.png", bbox_inches='tight', pad_inches=0, facecolor='black')
    plt.close(fig)
    return True

success_count = 0
for subj_id, df in tqdm(subject_dfs.items(), desc="Rendering Exaggerated Images"):
    lbl = labels[subj_id]
    if generate_exaggerated_signature(df, subj_id, lbl):
        success_count += 1

print(f"\n✓ Berhasil merender {success_count} gambar.")

Rendering Exaggerated Images:   0%|          | 0/70 [00:00<?, ?it/s]


✓ Berhasil merender 70 gambar.


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
import numpy as np
from tqdm.notebook import tqdm

In [13]:
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
import numpy as np
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tabular Setup
X_tab = df_macro[['fix_cx_std', 'gaze_x_range', 'regr_count', 'sacc_pv_mean',
                  'fix_disp_std', 'prog_amp_mean', 'vel_q95', 'fix_dur_q25', 'vel_median']].values
col_mask = ~np.isnan(X_tab).all(axis=0)
X_tab = np.nan_to_num(X_tab[:, col_mask], nan=0.0)
y_true = df_macro['label'].values

# Vision Setup
IMAGE_DIR = "/content/drive/MyDrive/TA_Project/dataset/improved_images/cnn_images"
vision_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_vision_dataset = datasets.ImageFolder(IMAGE_DIR, transform=vision_transforms)

# CHANGE 1: 35-Fold Cross Validation
K_FOLDS = 70
kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

final_tabular_preds = []
final_vision_preds = []
final_fusion_preds = []
actual_labels = [] # CHANGE 2: Must track labels because KFold shuffles the order

print(f"{'='*65}")
print(f"  PHASE 3: MULTIMODAL LATE FUSION ({K_FOLDS}-Fold CV)")
print(f"{'='*65}")

# Pre-load base weights so it doesn't download 35 times
base_weights = models.ResNet18_Weights.DEFAULT

# ==========================================
# 2. THE FUSION LOOP
# ==========================================
for train_idx, test_idx in tqdm(kfold.split(X_tab), total=K_FOLDS, desc="Evaluating Folds"):

    # Track the true labels for these specific test subjects
    actual_labels.extend(y_true[test_idx])

    # --- A. TABULAR MODEL (Left Brain) ---
    X_train_tab, X_test_tab = X_tab[train_idx], X_tab[test_idx]
    y_train_tab, y_test_tab = y_true[train_idx], y_true[test_idx]

    scaler = StandardScaler()
    X_train_tab_scaled = scaler.fit_transform(X_train_tab)
    X_test_tab_scaled = scaler.transform(X_test_tab)

    lr = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
    lr.fit(X_train_tab_scaled, y_train_tab)

    # Get array of probabilities for the test subjects
    probs_tab = lr.predict_proba(X_test_tab_scaled)[:, 1]

    # --- B. VISION MODEL (Right Brain) ---
    train_subset = Subset(full_vision_dataset, train_idx)
    test_subset = Subset(full_vision_dataset, test_idx)

    # Note: num_workers=2 speeds up Google Drive fetching slightly
    trainloader = DataLoader(train_subset, batch_size=8, shuffle=True, num_workers=2)
    testloader = DataLoader(test_subset, batch_size=2, shuffle=False)

    # Architect clean ResNet18
    model = models.resnet18(weights=base_weights)
    for param in model.parameters(): param.requires_grad = False
    for param in model.layer4.parameters(): param.requires_grad = True
    model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.fc.in_features, 2))
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam([
        {'params': model.layer4.parameters(), 'lr': 1e-4},
        {'params': model.fc.parameters(), 'lr': 1e-3}
    ], weight_decay=1e-4)

    # Fast train (10 epochs is enough for 35-fold)
    model.train()
    for epoch in range(10):
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # Extract probabilities
    model.eval()
    probs_vis = []
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
            # Grab class 1 (Dyslexic) probabilities and move to CPU list
            probs_vis.extend(probabilities[:, 1].cpu().tolist())

    # --- C. LATE FUSION ---
    # Iterate through the predictions (there will be 2 subjects per fold)
    for p_tab, p_vis in zip(probs_tab, probs_vis):
        prob_fusion = (p_tab + p_vis) / 2.0

        final_tabular_preds.append(1 if p_tab >= 0.5 else 0)
        final_vision_preds.append(1 if p_vis >= 0.5 else 0)
        final_fusion_preds.append(1 if prob_fusion >= 0.5 else 0)

# ==========================================
# 3. FINAL VERDICT
# ==========================================
acc_tab = accuracy_score(actual_labels, final_tabular_preds)
acc_vis = accuracy_score(actual_labels, final_vision_preds)
acc_fusion = accuracy_score(actual_labels, final_fusion_preds)

print(f"\nTabular Only Accuracy : {acc_tab:.4f}")
print(f"Vision Only Accuracy  : {acc_vis:.4f}")
print(f"FUSION ACCURACY       : {acc_fusion:.4f}")

  PHASE 3: MULTIMODAL LATE FUSION (70-Fold CV)


Evaluating Folds:   0%|          | 0/70 [00:00<?, ?it/s]


Tabular Only Accuracy : 0.8714
Vision Only Accuracy  : 0.6857
FUSION ACCURACY       : 0.7857


In [17]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneOut, cross_validate
from sklearn.pipeline import make_pipeline
import numpy as np

# Use your 9 proven optimal features
X_opt = df_macro[['fix_cx_std', 'gaze_x_range', 'regr_count', 'sacc_pv_mean',
                  'fix_disp_std', 'prog_amp_mean', 'vel_q95', 'fix_dur_q25', 'vel_median']].values

col_mask = ~np.isnan(X_opt).all(axis=0)
X_opt = np.nan_to_num(X_opt[:, col_mask], nan=0.0)
y_true = df_macro['label'].values

loo = LeaveOneOut()

# We test combinations of features (e.g., Feature A * Feature B, or Feature A^2)
pipelines = {
    "Linear LR (Baseline)": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight='balanced')
    ),
    "Polynomial LR (Degree 2)": make_pipeline(
        StandardScaler(),
        PolynomialFeatures(degree=2, interaction_only=False, include_bias=False),
        LogisticRegression(max_iter=5000, class_weight='balanced', C=0.5) # C=0.5 prevents overfitting the new features
    ),
    "SVM (Polynomial Degree 2)": make_pipeline(
        StandardScaler(),
        SVC(kernel='poly', degree=2, class_weight='balanced', C=1.0)
    ),
    "SVM (Polynomial Degree 3)": make_pipeline(
        StandardScaler(),
        SVC(kernel='poly', degree=3, class_weight='balanced', C=1.0)
    ),
    "SVM (Polynomial Degree 4)": make_pipeline(
        StandardScaler(),
        SVC(kernel='poly', degree=4, class_weight='balanced', C=1.0)
    ),
    "SVM (Polynomial Degree 5)": make_pipeline(
        StandardScaler(),
        SVC(kernel='poly', degree=5, class_weight='balanced', C=1.0)
    )
}

print(f"{'='*60}")
print(f"  PHASE 4: NON-LINEAR FEATURE EXPANSION (Target: 90%+)  ")
print(f"{'='*60}")

for name, pipe in pipelines.items():
    scores = cross_validate(
        pipe, X_opt, y_true, cv=loo,
        scoring={'accuracy': 'accuracy'},
        n_jobs=-1
    )
    acc = scores['test_accuracy'].mean()
    print(f"{name:<30} : {acc:.4f}  ({int(acc*len(y_true))}/{len(y_true)} correct)")

  PHASE 4: NON-LINEAR FEATURE EXPANSION (Target: 90%+)  
Linear LR (Baseline)           : 0.8714  (61/70 correct)
Polynomial LR (Degree 2)       : 0.7571  (53/70 correct)
SVM (Polynomial Degree 2)      : 0.5714  (40/70 correct)
SVM (Polynomial Degree 3)      : 0.6714  (46/70 correct)
SVM (Polynomial Degree 4)      : 0.6000  (42/70 correct)
SVM (Polynomial Degree 5)      : 0.5857  (41/70 correct)


In [18]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_validate
from sklearn.pipeline import make_pipeline
import numpy as np

# 1. Isolate the 9 proven optimal features
X_opt = df_macro[['fix_cx_std', 'gaze_x_range', 'regr_count', 'sacc_pv_mean',
                  'fix_disp_std', 'prog_amp_mean', 'vel_q95', 'fix_dur_q25', 'vel_median']].values

col_mask = ~np.isnan(X_opt).all(axis=0)
X_opt = np.nan_to_num(X_opt[:, col_mask], nan=0.0)
y_true = df_macro['label'].values

loo = LeaveOneOut()

# 2. Architect the Neural Network
# We use a tiny "bottleneck" architecture (16 neurons -> 8 neurons).
# Anything larger will instantly overfit 70 rows.
# High alpha (L2 penalty) prevents the network from memorizing the data.
mlp_pipeline = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(16, 8),
        activation='relu',
        solver='adam',
        alpha=0.5,           # Aggressive regularization
        max_iter=3000,
        random_state=42
    )
)

print(f"{'='*60}")
print(f"  PHASE 5: TABULAR DEEP LEARNING (MLP NEURAL NETWORK)  ")
print(f"{'='*60}")

scores = cross_validate(
    mlp_pipeline, X_opt, y_true, cv=loo,
    scoring={'accuracy': 'accuracy'},
    n_jobs=-1,
    return_train_score=True
)

acc = scores['test_accuracy'].mean()
gap = scores['train_accuracy'].mean() - acc

print(f"MLP Accuracy     : {acc:.4f}  ({int(acc*len(y_true))}/{len(y_true)} correct)")
print(f"Overfitting Gap  : {gap:+.4f}")

  PHASE 5: TABULAR DEEP LEARNING (MLP NEURAL NETWORK)  
MLP Accuracy     : 0.8429  (59/70 correct)
Overfitting Gap  : +0.1522


In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import numpy as np
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. EXTRACT RAW TIME-SERIES
# ==========================================
max_seq_len = 1500 # Pad short readings, truncate excessively long ones
X_raw = []
y_raw = []
subject_ids = list(subject_dfs.keys())

print("Formatting sequential data...")
for subj_id in subject_ids:
    df = subject_dfs[subj_id]
    x = df[['gaze_x_left', 'gaze_x_right']].mean(axis=1).values
    y = df[['gaze_y_left', 'gaze_y_right']].mean(axis=1).values

    valid = ~(np.isnan(x) | np.isnan(y))
    x, y = x[valid], y[valid]

    seq = np.column_stack((x, y))

    # Pad or truncate to uniform length
    if len(seq) > max_seq_len:
        seq = seq[:max_seq_len]
    else:
        seq = np.vstack((seq, np.zeros((max_seq_len - len(seq), 2))))

    X_raw.append(seq)
    y_raw.append(labels[subj_id])

X_seq = np.array(X_raw, dtype=np.float32)
y_seq = np.array(y_raw, dtype=np.int64)

# ==========================================
# 2. LSTM ARCHITECTURE
# ==========================================
class GazeLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        # 1 LSTM layer, heavy dropout to prevent memorizing the 70 subjects
        self.lstm = nn.LSTM(input_size=2, hidden_size=32, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(32, 2)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        out = self.dropout(hn[-1])
        return self.fc(out)

# ==========================================
# 3. 70-FOLD LOOCV
# ==========================================
loo = LeaveOneOut()
lstm_preds = []

print(f"\n{'='*60}")
print(f"  PHASE 6: RAW SEQUENCE LSTM (70-Fold LOOCV)")
print(f"{'='*60}")

for train_idx, test_idx in tqdm(loo.split(X_seq), total=len(X_seq), desc="Training Folds"):
    X_train, X_test = X_seq[train_idx], X_seq[test_idx]
    y_train, y_test = y_seq[train_idx], y_seq[test_idx]

    # Scale coordinates per fold to prevent data leakage
    scaler = StandardScaler()
    X_train_flat = X_train.reshape(-1, 2)
    X_test_flat = X_test.reshape(-1, 2)

    X_train_scaled = scaler.fit_transform(X_train_flat).reshape(X_train.shape)
    X_test_scaled = scaler.transform(X_test_flat).reshape(X_test.shape)

    X_train_t = torch.tensor(X_train_scaled).to(device)
    X_test_t = torch.tensor(X_test_scaled).to(device)
    y_train_t = torch.tensor(y_train).to(device)

    model = GazeLSTM().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    model.train()
    # Fast training loop
    for epoch in range(30):
        optimizer.zero_grad()
        out = model(X_train_t)
        loss = criterion(out, y_train_t)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        out = model(X_test_t)
        pred = torch.argmax(out, dim=1).item()
        lstm_preds.append(pred)

acc = accuracy_score(y_seq, lstm_preds)
print(f"\nLSTM Final Accuracy: {acc:.4f} ({int(acc*len(y_seq))}/{len(y_seq)} correct)")

Formatting sequential data...

  PHASE 6: RAW SEQUENCE LSTM (70-Fold LOOCV)


Training Folds:   0%|          | 0/70 [00:00<?, ?it/s]


LSTM Final Accuracy: 0.7143 (50/70 correct)
